# 05. O investidor (utilidade CRRA)
Classe Investidor: a utilidade, a carteira e o consumo. Usa o nucleo e o mercado, que já estão prontos. Requisitos F5, F6, F8 e F9.

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np
from app import nucleo
from app.mercado import RendaFixa, RendaVariavel

## Desenvolvimento

A classe abaixo foi escrita e testada aqui, e depois foi para app/agente.py.

In [3]:
class Investidor:
    """Agente CRRA com decisão de consumo e portfólio. (F5)"""

    def __init__(self, gamma: float, beta: float, w0: float, horizonte: int) -> None:
        if gamma <= 0:
            raise ValueError("γ (aversão ao risco) deve ser > 0.")
        if not 0.0 < beta < 1.0:
            raise ValueError("β (fator de desconto) deve estar em (0, 1).")
        if w0 <= 0:
            raise ValueError("W₀ (riqueza inicial) deve ser > 0.")
        if horizonte < 1:
            raise ValueError("horizonte T deve ser ≥ 1.")
        self.gamma = float(gamma)
        self.beta = float(beta)
        self.w0 = float(w0)
        self.horizonte = int(horizonte)
        self._alpha_star: np.ndarray | None = None
        self._phi_hat: float | None = None
        self._A: np.ndarray | None = None

    # utilidade CRRA (F5)
    def utilidade(self, c):
        """u(c) = c^(1−γ)/(1−γ) (ou ln c se γ=1). (F5)"""
        c = np.asarray(c, dtype=float)
        if np.isclose(self.gamma, 1.0):
            return np.log(c)
        return c ** (1.0 - self.gamma) / (1.0 - self.gamma)

    def utilidade_marginal(self, c):
        """u'(c) = c^(−γ). (F5)"""
        return np.asarray(c, dtype=float) ** (-self.gamma)

    # decisao de carteira e de consumo (F6, F8, F9)
    def carteira_otima(self, mercado: RendaVariavel, rf: float, *,
                       n_scenarios: int = 100_000, seed: int | None = 42,
                       **opts) -> np.ndarray:
        """Carteira otima, resolvendo G(alpha)=0. (F6)

        Sorteia os cenarios do mercado (que vem liquidos) e converte pra fator
        bruto. O alpha e sempre livre, pode ser negativo e pode passar de 1.

        O que vier em opts vai direto pro nucleo.resolver_alpha_otimo
        (tol, maxiter, alpha0).
        """
        r = mercado.amostrar(n_scenarios, seed=seed)
        rf_bruto = 1.0 + rf
        R = np.maximum(1.0 + r, 0.0)  # resp. limitada do ativo: preço não fica < 0
        alpha = nucleo.resolver_alpha_otimo(R, rf_bruto, self.gamma, **opts)
        self._alpha_star = alpha
        self._phi_hat = nucleo.phi_chapeu(alpha, R, rf_bruto, self.gamma)
        return alpha

    def fracoes_consumo(self) -> np.ndarray:
        """Frações de consumo theta_t = A_t^(−1/γ), t=0..T. (F8, F9)

        Requer carteira_otima(...) chamado antes (usa o Phi_chapeu guardado).
        """
        if self._phi_hat is None:
            raise RuntimeError(
                "chame carteira_otima() antes de fracoes_consumo(): "
                "as fracoes de consumo dependem do phi, que sai da carteira."
            )
        self._A = nucleo.recorrencia_A(self._phi_hat, self.beta, self.gamma, self.horizonte)
        return nucleo.fracoes_consumo(self._A, self.gamma)

    @property
    def alpha_star(self) -> np.ndarray | None:
        """Última carteira ótima alpha* calculada (ou None)."""
        return self._alpha_star

    @property
    def phi_hat(self) -> float | None:
        """Phi_chapeu da última política ótima (ou None)."""
        return self._phi_hat

    @property
    def coeficientes_A(self) -> np.ndarray | None:
        """Os coeficientes A_t da ultima recorrencia (Etapa 3), ou None.

        Ficam guardados porque a funcao valor (F11) precisa deles, e sem isso
        eles teriam que ser calculados de novo la na frente.
        """
        return self._A

**Teste**: a utilidade CRRA e a decisão ótima, ou seja, carteira e consumo.

In [4]:
import pandas as pd
inv = Investidor(gamma=5.0, beta=0.96, w0=1.0, horizonte=12)

print('u(2):', inv.utilidade(2.0), "| u'(2):", inv.utilidade_marginal(2.0))

u(2): -0.015625 | u'(2): 0.03125


In [5]:
assert np.isclose(inv.utilidade(2.0), 2.0**(-4)/(-4)) and np.isclose(inv.utilidade_marginal(2.0), 2.0**(-5))

In [6]:
rng = np.random.default_rng(7)
ruido = rng.normal(0,0.06,300)
ruido -= ruido.mean() 
ret = pd.DataFrame({'data': pd.date_range('2000-01',periods=300,freq='MS').strftime('%Y-%m'),'ibov':0.015+ruido}) 
alpha = inv.carteira_otima(RendaVariavel(ret), RendaFixa(0.10).retorno_livre_risco(), n_scenarios=40_000, seed=1)

print('carteira_otima:', alpha, '| alpha_star:', inv.alpha_star, '| phi_hat:', inv.phi_hat)

carteira_otima: [0.43327271] | alpha_star: [0.43327271] | phi_hat: 0.9633213419184827


In [7]:
theta = inv.fracoes_consumo(); print('theta:', theta)

theta: [0.08434563 0.09068584 0.09818248 0.10718248 0.11818697 0.13194762
 0.14964566 0.17324977 0.20630361 0.25589447 0.33855939 0.50390943
 1.        ]


In [8]:
assert np.isclose(theta[-1], 1.0) and np.all(np.diff(theta) > 0)